# Vision Foundation Models (VFMs): One Backbone, Many Applications

A **vision foundation model** is a neural network pretrained once, on huge unlabeled data, then
**frozen** and reused for many different computer-vision downstream tasks, often with no
task-specific training at all, or with a tiny head trained in seconds. We'll use
[DINOv2](https://dinov2.metademolab.com) to demonstrate this. DINOv2 is a landmark vision foundation
model introduced by Meta AI. Unlike traditional vision models trained on millions of
human-annotated images, DINOv2 (as well as DINO and DINOv3) learns rich visual representations
purely from unlabeled data — replacing supervised learning with **self-supervised learning** on
Vision Transformers (ViT).

<p align="center">
  <img src="./vit.png" width="600">
</p>

> Oquab, Maxime, et al. "DINOv2: Learning robust visual features without supervision." arXiv preprint arXiv:2304.07193 (2023). [link](https://arxiv.org/pdf/2304.07193)

## imports and Plotting helpers

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
def to_rgb(pixel_tensor):
    '''Convert a normalized (3, H, W) tensor into a displayable (H, W, 3) numpy image.'''
    img = pixel_tensor.detach().cpu().permute(1, 2, 0).numpy()
    return (img - img.min()) / (img.max() - img.min() + 1e-8)


def show_image_grid(images, titles=None, ncols=5, figsize_per_img=(3, 3), suptitle=None):
    '''Simple grid of PIL images (or numpy arrays) with optional titles.'''
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_img[0] * ncols, figsize_per_img[1] * nrows))
    axes = np.atleast_1d(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i])
            if titles is not None:
                ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
    if suptitle:
        plt.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.show()


def overlay_heatmap(ax, base_img, heatmap, cmap="jet", alpha=0.55, vmin=0.0, vmax=1.0):
    '''Draw base_img with a similarity/cluster heatmap overlaid, scaled to the image size.'''
    
    ax.imshow(base_img)
    if heatmap.dtype == np.int32 or heatmap.dtype == np.int64:
        # integer cluster labels: use discrete colormap
        im = ax.imshow(heatmap, cmap=cmap, alpha=alpha, extent=[0, base_img.shape[1], base_img.shape[0], 0], interpolation="nearest")
    else:
        # continuous values: use bilinear interpolation
        im = ax.imshow(heatmap, cmap=cmap, alpha=alpha, vmin=vmin, vmax=vmax,
                    extent=[0, base_img.shape[1], base_img.shape[0], 0], interpolation="bilinear")
    ax.axis("off")
    return im


def plot_correspondences(image_a, image_b, points_a, points_b, scores=None, title=None):
    '''Draw matched point pairs between two images as numbered, color-coded markers.'''
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_a); axes[0].axis("off")
    axes[1].imshow(image_b); axes[1].axis("off")
    colors = plt.cm.tab10(np.linspace(0, 1, len(points_a)))
    for i, ((xa, ya), (xb, yb)) in enumerate(zip(points_a, points_b)):
        for ax, (x, y) in [(axes[0], (xa, ya)), (axes[1], (xb, yb))]:
            ax.scatter(x, y, color=colors[i], s=100, edgecolors="white", linewidth=2, zorder=5)
            ax.text(x, y, str(i + 1), color="black", fontsize=9, fontweight="bold",
                    ha="center", va="center", zorder=6)
    if title:
        plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


# Part 1: Image-Level Features


Let's start with the dataset we'll use to demonstrate DINOv2's image-level features:
**Imagenette**, a small, curated subset of ImageNet with 10 easily distinguishable classes.


In [ ]:
import torchvision.datasets as datasets

IMAGENETTE_ROOT = "./data/imagenette/" 

dataset = datasets.Imagenette(root=IMAGENETTE_ROOT, split="val", size="full", download=True)
class_names = [name[0] for name in dataset.classes]
print(f"Number of images in the dataset: {len(dataset)}")
print(f"Classes in the dataset: {class_names}")

num_samples = 5
sample_ids = np.random.choice(len(dataset), size=num_samples, replace=False)
sample_images = [dataset[i][0] for i in sample_ids]
sample_titles = [f"Class: {class_names[dataset[i][1]]}" for i in sample_ids]
show_image_grid(sample_images, sample_titles, ncols=num_samples, figsize_per_img=(4, 4))


Now we load DINOv2 through the Hugging Face `transformers` library.


In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINO_MODEL_NAME = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
dino = AutoModel.from_pretrained(DINO_MODEL_NAME).to(device).eval()
for p in dino.parameters():
    p.requires_grad = False

PATCH_SIZE = dino.config.patch_size
HIDDEN_DIM = dino.config.hidden_size
NUM_REGISTER_TOKENS = getattr(dino.config, "num_register_tokens", 0)

print(f"Model: {DINO_MODEL_NAME}")
print(f"Patch size: {PATCH_SIZE}, hidden dim: {HIDDEN_DIM}, register tokens: {NUM_REGISTER_TOKENS}")


To get a **global** representation of an image, we use the ViT's `[CLS]` token. During the
forward pass, this token continuously aggregates context from every patch via self-attention, so its
final hidden state ends up as a fixed-length summary of the whole image.

**Task 1:** compute DINOv2's image-level (`[CLS]`) feature vector for a given input image.


In [ ]:
def get_global_feature(pil_img):
    '''Extract and L2-normalize the global [CLS] feature vector for one PIL image.
    Returns a tensor of shape (1, HIDDEN_DIM).'''
    cls_token = None  # placeholder for the output feature vector
    
    # TODO: prepare the input for the DINO model
    

    # TODO: run the DINO model in inference mode to get the output features
    

    # TODO: extract the [CLS] token from the output and L2-normalize it
        

    return cls_token


sample_img = dataset[sample_ids[0]][0]
global_feature = get_global_feature(sample_img)
print(f"Global feature shape: {global_feature.shape}")   # (1, 768)


Using the ``get_global_feature `` we can compute the feature vector for *every* image in the dataset. This is the "index" we'll reuse for retrieval, clustering, and classification below.


In [ ]:
dataset_features, dataset_labels = [], []
for img, label in tqdm(dataset, desc="Extracting features"):
    dataset_features.append(get_global_feature(img))
    dataset_labels.append(label)

dataset_features = torch.vstack(dataset_features)
dataset_labels = torch.tensor(dataset_labels)
print(f"Extracted features shape: {dataset_features.shape}, labels shape: {dataset_labels.shape}")


Now let's use **t-SNE** to visualize these high-dimensional features in 2D. t-SNE performs non-linear dimensionality reduction while preserving *local* distances. So, nearby points in the plot
were nearby in the original 768-dim space too, which lets us spot clusters visually.


In [ ]:
def plot_tsne(features, labels, class_names, title):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords_2d = tsne.fit_transform(features.cpu().numpy())
    labels_np = labels.cpu().numpy()

    plt.figure(figsize=(10, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    for idx, name in enumerate(class_names):
        mask = labels_np == idx
        plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], label=name,
                    color=colors[idx], s=40, alpha=0.85, edgecolors="none")
    plt.title(title, fontsize=13)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

plot_tsne(dataset_features, dataset_labels, class_names,
          "Unsupervised Semantic Topology of Imagenette (t-SNE on DINOv2 [CLS])")


**What do you observe?** Images from the same class should form distinct, well-separated clusters, with zero supervision anywhere in the DINOv2 pretraining pipeline. Now let's zoom into
*one* cluster: if we sub-cluster within a single class, what do we find?


In [ ]:
def show_subclusters(target_class_idx, n_subclusters=6, samples_per_subcluster=6):
    class_mask = (dataset_labels.cpu().numpy() == target_class_idx)
    class_indices = np.where(class_mask)[0]
    class_features = dataset_features[class_mask].cpu().numpy()

    kmeans = KMeans(n_clusters=n_subclusters, random_state=42, n_init=10)
    sub_labels = kmeans.fit_predict(class_features)

    images, titles = [], []
    for k in range(n_subclusters):
        members = np.where(sub_labels == k)[0]
        center = kmeans.cluster_centers_[k]
        distances = np.linalg.norm(class_features[members] - center, axis=1)
        closest = members[np.argsort(distances)[:samples_per_subcluster]]
        for rank, m in enumerate(closest):
            images.append(dataset[class_indices[m]][0])
            titles.append(f"Subcluster {k+1}" if rank == 0 else "")

    show_image_grid(images, titles, ncols=samples_per_subcluster,
                     suptitle=f"Representative images per subcluster: '{class_names[target_class_idx]}'")

show_subclusters(target_class_idx=4)   # 4 = 'church' -- try other indices, see class_names above


Within each subcluster, images with similar visual style, color scheme, or pose land close together. DINOv2 is capturing fine-grained similarity, not just coarse category membership.

A good representation should also stay stable under **nuisance** changes (crops, color, blur) while still reacting to genuine **semantic** changes (a different object entirely). Let's test that
directly: original vs. augmented, same class/different image, and different class.


In [ ]:
import torchvision.transforms.functional as TF

def build_invariance_set(reference_class_idx):
    ref_idx = np.random.choice(np.where(dataset_labels.numpy() == reference_class_idx)[0])
    ref_img, ref_label = dataset[ref_idx]

    same_class_idxs = np.where(dataset_labels.numpy() == ref_label)[0]
    same_idx = np.random.choice(same_class_idxs[same_class_idxs != ref_idx])
    same_class_img, _ = dataset[same_idx]

    diff_idx1, diff_idx2 = np.random.choice(np.where(dataset_labels.numpy() != ref_label)[0], size=2, replace=False)
    diff_img1, diff_label1 = dataset[diff_idx1]
    diff_img2, diff_label2 = dataset[diff_idx2]

    return {
        "Original (ref)": ref_img,
        "Horizontal flip": TF.hflip(ref_img),
        "Color jitter": TF.adjust_contrast(TF.adjust_brightness(ref_img, 2.0), 0.6),
        "Grayscale": TF.to_grayscale(ref_img, num_output_channels=3),
        "Gaussian blur": TF.gaussian_blur(ref_img, kernel_size=[25, 25], sigma=[3.0, 3.0]),
        "Rotation (45deg)": TF.rotate(ref_img, angle=45),
        f"Same class ({class_names[ref_label]})": same_class_img,
        f"Diff class ({class_names[diff_label1]})": diff_img1,
        f"Diff class ({class_names[diff_label2]})": diff_img2,
    }, ref_label


def plot_invariance_profile(edit_dict, ref_label):
    labels_list = list(edit_dict.keys())
    imgs = [img.convert("RGB") for img in edit_dict.values()]

    inputs = processor(images=imgs, return_tensors="pt").to(device)
    with torch.no_grad():
        cls_tokens = F.normalize(dino(**inputs).last_hidden_state[:, 0, :], dim=-1)
    sims = (cls_tokens @ cls_tokens[0:1].T).squeeze(-1).cpu().numpy()

    show_image_grid(list(edit_dict.values()), labels_list, ncols=3,
                     suptitle=f"Reference class: '{class_names[ref_label]}'")

    colors = ["#1f77b4"] * 6 + ["#2ca02c"] + ["#d62728"] * 2
    plt.figure(figsize=(9, 4))
    plt.barh(labels_list[::-1], sims[::-1], color=colors[::-1], edgecolor="black", alpha=0.85)
    plt.xlim(0, 1.05)
    plt.xlabel("Cosine similarity to reference [CLS]")
    plt.tight_layout()
    plt.show()


edit_dict, ref_label = build_invariance_set(reference_class_idx=1) 
plot_invariance_profile(edit_dict, ref_label)


DINOv2 embeddings stay close under nuisance edits (flip, color, blur, rotation) while dropping
off for genuine semantic changes (same class/different image drops a bit; different class drops a
lot). This is exactly the property that makes DINOv2 useful for **content-based image retrieval
(CBIR)**: find images that are visually/semantically similar to a query, without any labels.

**Task 2:** build a simple retrieval system — given a query image, return the top-*k* most similar
images in the dataset by cosine similarity.

$$Sim(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$


In [ ]:
def retrieve_top_k(query_embedding, gallery_embeddings, k=5):
    '''Rank gallery items by cosine similarity to a (1, dim) query embedding.
    Returns (top_k_indices, all_similarities).'''
    
    
    # TODO: compute the cosine similarities between the query embedding and all gallery embeddings
    similarities = ...
    
    # TODO: find the indices of the top-k most similar gallery items
    top_k_indices = ...
    

    return top_k_indices.cpu().numpy(), similarities.cpu().numpy()


def plot_retrieval(query_img, top_k_indices, sims):
    images = [query_img] + [dataset[i][0] for i in top_k_indices]
    titles = ["QUERY"] + [f"#{r+1} {class_names[dataset[i][1]]}\nsim={sims[i]:.3f}"
                           for r, i in enumerate(top_k_indices)]
    show_image_grid(images, titles, ncols=len(images))


import requests
from io import BytesIO
QUERY_IMAGE_URL = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQfT__VyBE46WNJHZFCM_vMtA-36xx-52pKyzvi-y21wQ&s=10"  # <-- paste your URL here

# Fetch the image from the URL
response = requests.get(QUERY_IMAGE_URL)
response.raise_for_status() # This ensures you get an error if the download fails

# Open the image from the downloaded bytes
query_img = Image.open(BytesIO(response.content))
K = 5
query_embedding = get_global_feature(query_img)
top_k_indices, sims = retrieve_top_k(query_embedding, dataset_features, k=K)
plot_retrieval(query_img, top_k_indices, sims)


DINOv2 also makes a strong **classification backbone**: freeze it, add a linear layer on top of
the `[CLS]` token, and train only that layer. Here's the PyTorch module for that pattern (useful if
you want a trainable head as part of a larger pipeline):


In [ ]:
class DINOv2ImageClassifier(nn.Module):
    '''Frozen DINOv2 backbone + a single trainable linear layer (linear probing).'''

    def __init__(self, dino_model, num_classes, freeze_backbone=True):
        super().__init__()
        self.dino = dino_model
        if freeze_backbone:
            for p in self.dino.parameters():
                p.requires_grad = False
        self.classifier = nn.Linear(dino_model.config.hidden_size, num_classes)

    def forward(self, pixel_values):
        with torch.no_grad():
            cls_token = self.dino(pixel_values=pixel_values).last_hidden_state[:, 0, :]
        return self.classifier(cls_token)



# Part 2: Image Patch Features

So far we've used DINOv2's *image-level* features: robust to nuisance variation, useful for
retrieval, and a strong classification backbone. These capabilities exist in other modern vision
models too. What's distinctive about DINOv2 is its **dense patch features**. Instead of one vector
per image, we get one vector *per patch*, which tells us not just *what* is in the image but *where*.

For this part we'll use the COCO dataset (with its segmentation annotations, which we'll reuse for
foreground masking later).


In [ ]:
import os
import requests
import zipfile
from tqdm import tqdm

def download_and_extract(url, filename, target_dir="./data/coco/"):
    # 1. Create the target directory if it doesn't already exist
    os.makedirs(target_dir, exist_ok=True)
    
    # Create the full path for the zip file (e.g., ./data/coco/val2017.zip)
    zip_path = os.path.join(target_dir, filename)
    
    # 2. Download the file into the target directory
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    print(f"Downloading {filename} to {target_dir}...")
    with open(zip_path, 'wb') as file, tqdm(
        desc=filename,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)
            
    # 3. Extract the zip file into the target directory
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Extractall takes the destination path as an argument
        zip_ref.extractall(target_dir) 
    print("Done!\n")

# Run the downloads
download_and_extract("http://images.cocodataset.org/zips/val2017.zip", "val2017.zip")
download_and_extract("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "annotations_trainval2017.zip")

In [ ]:
import random
from torchvision.datasets import CocoDetection

COCO_IMAGES = "./data/coco/val2017"                   
COCO_ANNOTATIONS = "./data/coco/annotations/instances_val2017.json"

coco_dataset = CocoDetection(root=COCO_IMAGES, annFile=COCO_ANNOTATIONS)

sample_indices = random.sample(range(len(coco_dataset)), 5)
colors = plt.cm.tab20.colors

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, idx in zip(axes, sample_indices):
    img, anns = coco_dataset[idx]
    obj_names = [coco_dataset.coco.loadCats(a["category_id"])[0]["name"] for a in anns]
    rgba_mask = np.zeros((img.height, img.width, 4), dtype=np.float32)
    for i, a in enumerate(anns):
        m = coco_dataset.coco.annToMask(a) > 0
        rgba_mask[m, :3] = colors[i % len(colors)]
        rgba_mask[m, 3] = 0.55
    ax.imshow(img)
    ax.imshow(rgba_mask)
    ax.set_title(", ".join(set(obj_names)) or "No objects", fontsize=10)
    ax.axis("off")
plt.suptitle("COCO samples with per-instance segmentation masks", fontsize=13)
plt.tight_layout()
plt.show()


**Task 3:** implement patch-feature extraction. This is the patch-level equivalent of
`get_global_feature` above. Instead of keeping only the `[CLS]` token, we keep every patch token
and arrange them back into their original spatial grid.


In [ ]:
@torch.no_grad()
def get_patch_features(pil_img):
    '''Extract dense DINOv2 patch features for one image.
    Returns (patch_grid: (H_patches, W_patches, hidden_dim), pixel_values: (3, H, W)).'''
    
    # prepare the input for the DINO model
    inputs = processor(images=pil_img, return_tensors="pt", do_resize=False, do_center_crop=False).to(device)
    
    # TODO: run the DINO model in inference mode to get the output features
    

    # TODO: extract the patch tokens (drop [CLS] and any register tokens) and L2-normalize them
    

    # TODO: reshape the patch tokens into a 2D grid (H_patches, W_patches, hidden_dim) and return it along with the original pixel values
    
    patch_grid = ... # output placeholder
    

    return patch_grid, inputs["pixel_values"][0]


sample_img, _ = coco_dataset[0]
patch_grid, pixel_tensor = get_patch_features(sample_img)
print(f"Pixel tensor shape : {pixel_tensor.shape}")
print(f"Patch grid shape   : {patch_grid.shape}  ({patch_grid.shape[0]}x{patch_grid.shape[1]} patches)")


Let's cluster an image's patch features and visualize the clusters spatially. If patches
with similar visual/semantic content are represented similarly, clustering should recover meaningful
regions with zero labels.


In [ ]:
def cluster_patch_grid(patch_grid, n_clusters):
    '''K-means over patch features. Returns a (H, W) integer cluster-label map.'''
    h, w, dim = patch_grid.shape
    flat = patch_grid.reshape(-1, dim).cpu().numpy()
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_map = kmeans.fit_predict(flat).reshape(h, w)
    
    return cluster_map


def plot_multiscale_clusters(image, patch_grid, pixel_tensor, k_values=(2, 4, 8)):
    disp_img = to_rgb(pixel_tensor)
    fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(18, 4.5))
    axes[0].imshow(disp_img)
    axes[0].set_title(f"Input\n{patch_grid.shape[0]}x{patch_grid.shape[1]} patches", fontsize=11)
    axes[0].axis("off")
    for i, k in enumerate(k_values):
        cluster_map = cluster_patch_grid(patch_grid, n_clusters=k)
        overlay_heatmap(axes[i + 1], disp_img, cluster_map, cmap="tab10", alpha=0.55,
                         vmin=0, vmax=max(k_values) - 1)
        axes[i + 1].set_title(f"k = {k}", fontsize=11)
    plt.suptitle("Hierarchical patch clustering (unsupervised)", fontsize=13)
    plt.tight_layout()
    plt.show()


sample_idx = np.random.randint(0, len(coco_dataset))
sample_img, _ = coco_dataset[sample_idx]
patch_grid, pixel_tensor = get_patch_features(sample_img)
plot_multiscale_clusters(sample_img, patch_grid, pixel_tensor)


**What do you observe** as *k* increases? Typically: `k=2` gives a coarse foreground/background
split, `k=4` starts separating major object regions, `k=8` breaks things down into sub-parts and
textures — all from the same frozen features, just asking k-means for a different number of groups.

Go one step further. Instead of clustering within one image, use the patch features of a specific object in image A to *find that object in image B* (zero-shot object localization via
mask query transfer).

**Task 4:** Compute a cosine-similarity heatmap between the patches of a query object and all patches in a target image.


In [ ]:
def build_query_embedding(patch_grid, full_res_mask):
    '''Average the patch features that fall under a full-resolution binary mask.
    Returns the query patch vectors, shape (N_masked_patches, dim).'''
    h, w, _ = patch_grid.shape
    mask_tensor = torch.from_numpy(full_res_mask).float().unsqueeze(0).unsqueeze(0)
    patch_mask = F.interpolate(mask_tensor, size=(h, w), mode="nearest").squeeze() > 0.5
    return patch_grid[patch_mask]


def similarity_heatmap(query_patches, target_patch_grid, top_k=5):
    '''For every patch in target_patch_grid, similarity to its top-k best-matching query patches.
    Returns a (H, W) heatmap in [-1, 1].'''
    
    # TODO: compute the cosine similarity between each target patch and all query patches (H, W, N_query)    
    
    # TODO: find the top-k similarities for each target patch and average them to create a heatmap    
    heatmap = ... # output placeholder
    
    return heatmap.cpu().numpy()


# Example: find the query object (a zebra) from image A in image B
example = 0
img_a_idx =  [4828, 4625, 4182][example]   # Image containing the query object mask
img_b_idx = [435, 4556, 4805][example]  # Target image to search in
img_a, anns_a = coco_dataset[img_a_idx]
img_b, anns_b = coco_dataset[img_b_idx]

selected_ann = anns_a[0]
category_name = coco_dataset.coco.loadCats(selected_ann["category_id"])[0]["name"]
full_mask_a = coco_dataset.coco.annToMask(selected_ann) > 0

patch_grid_a, pixel_a = get_patch_features(img_a)
patch_grid_b, pixel_b = get_patch_features(img_b)
query_patches = build_query_embedding(patch_grid_a, full_mask_a)
heatmap = similarity_heatmap(query_patches, patch_grid_b, top_k=5)

disp_a, disp_b = to_rgb(pixel_a), to_rgb(pixel_b)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(disp_a); axes[0].imshow(full_mask_a, cmap="spring", alpha=0.5)
axes[0].set_title(f"Image A: query object ('{category_name}')", fontsize=11); axes[0].axis("off")
axes[1].imshow(disp_b); axes[1].set_title("Image B: search target", fontsize=11); axes[1].axis("off")
im = overlay_heatmap(axes[2], disp_b, heatmap, cmap="jet", alpha=0.55, vmin=0.0, vmax=1.0)
axes[2].set_title(f"Similarity to '{category_name}' query", fontsize=11)
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle("Zero-shot object localization via mask query transfer", fontsize=13)
plt.tight_layout()
plt.show()


The localization demo above matched a *region* (a whole object's patches) from one image to another.
We can go more fine-grained: pick a handful of individual **landmark points** on a reference image
(eyes, nose, chin) and find each one's best-matching patch in a *different pose* of the same object,
i.e. ask DINOv2 to track specific points, not just "where is this object," across viewpoint change.


In [ ]:

PORTRAIT_PATH = "./portrait.jpg"
N_COLS, N_ROWS = 8, 3
# Resize each crop to a multiple of PATCH_SIZE so get_patch_features works cleanly
PORTRAIT_SIZE = 224  # 14 × 16 — standard DINOv2 input size

full_img = Image.open(PORTRAIT_PATH).convert("RGB")
W, H = full_img.size
cell_w = W // N_COLS
cell_h = H // N_ROWS

portrait_imgs = []
for row in range(N_ROWS):
    for col in range(N_COLS):
        crop = full_img.crop((col * cell_w, row * cell_h,
                               (col + 1) * cell_w, (row + 1) * cell_h))
        portrait_imgs.append(crop.resize((PORTRAIT_SIZE, PORTRAIT_SIZE), Image.LANCZOS))

N = len(portrait_imgs)  # 24

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(2.5 * N_COLS, 3 * N_ROWS))
for idx, img in enumerate(portrait_imgs):
    r, c = divmod(idx, N_COLS)
    axes[r, c].imshow(img)
    axes[r, c].set_title(f"[{r},{c}]", fontsize=9)
    axes[r, c].axis('off')
plt.suptitle("Portrait2 – 24 Face Sub-Images (3 rows × 8 columns)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Each 224×224 image → 16×16 = 256 patch tokens of shape (16, 16, 768)
portrait_grids = []    # list of (16, 16, hidden_dim) tensors
portrait_pixels = []   # list of (3, 224, 224) tensors for display

for img in tqdm(portrait_imgs, desc="Extracting patch features"):
    grid, pix = get_patch_features(img)
    portrait_grids.append(grid)
    portrait_pixels.append(pix)

h_p, w_p, dim = portrait_grids[0].shape
print(f"Extracted features for {N} portraits | patch grid: {h_p}×{w_p} | dim: {dim}")

**Task 5:** implement landmark matching. given a reference patch vector for one landmark, find
its best match (by cosine similarity) in a target image's patch grid. This is the same nearest-
neighbor idea as `similarity_heatmap` above, just returning a single best point instead of a full
heatmap.

In [ ]:
from matplotlib.patches import Rectangle

# Portrait – Landmark-Based Patch Correspondence Mosaic
ROW_REF = 1   # reference row    (0–2)
COL_REF = 0   # reference column (0–7)

# Grid is 16×16; patch (r,c) covers pixels [r*14:(r+1)*14, c*14:(c+1)*14]
LANDMARKS = {"forehead": (3,  7), "l-eye":    (6,  5), "r-eye":    (6, 10), "nose":     (7,  7), "mouth":  (9, 7), "chin": (11, 7),}
SIM_THRESHOLD = 0.5   # drop landmarks whose best match falls below this

ref_idx    = ROW_REF * N_COLS + COL_REF
fa_ref_cpu = portrait_grids[ref_idx].reshape(-1, dim).cpu()   # (256, dim)

lm_names = list(LANDMARKS.keys())
lm_flat  = [r * w_p + c for r, c in LANDMARKS.values()]   # flat patch indices

cmap     = plt.cm.tab10
lm_color = {name: cmap(i % 10) for i, name in enumerate(lm_names)}

# Extract one feature vector per landmark
fa_lm = fa_ref_cpu[lm_flat]   # (N_lm, dim)

# ---------------------------------------------------------------
# For each non-reference image: best-matching patch per landmark
# all_matches[j] = [(lm_name, pa_flat, pb_flat, score), ...]
# ---------------------------------------------------------------
all_matches = {}

for j in range(N):
    if j == ref_idx:
        continue
    fb_cpu = portrait_grids[j].reshape(-1, dim).cpu()
    
    # TODO: compute cosine similarity between fa_lm and fb_cpu, find best-matching patch for each landmark
    sim = ...  # (N_lm, 256)
    best_pb  = ...         # (N_lm,)
    best_score = ...     # (N_lm,)

    all_matches[j] = [
        (lm_names[k], lm_flat[k], best_pb[k].item(), best_score[k].item())
        for k in range(len(lm_names))
        if best_score[k].item() >= SIM_THRESHOLD
    ]

# ---------------------------------------------------------------
# Draw mosaic
# ---------------------------------------------------------------
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(3.2 * N_COLS, 3.8 * N_ROWS))

for idx in range(N):
    r, c = divmod(idx, N_COLS)
    ax   = axes[r, c]
    disp = to_rgb(portrait_pixels[idx])
    ax.imshow(disp)
    ax.set_xticks([]); ax.set_yticks([])

    if idx == ref_idx:
        rect = Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                          fill=False, edgecolor='#FF2222',
                          linewidth=6, clip_on=False)
        ax.add_patch(rect)
        ax.set_title("● REFERENCE", fontsize=7,
                     color='#CC0000', fontweight='bold', pad=2)
        for name, pa_flat in zip(lm_names, lm_flat):
            ya, xa = pa_flat // w_p, pa_flat % w_p
            px, py = (xa + 0.5) * PATCH_SIZE, (ya + 0.5) * PATCH_SIZE
            ax.scatter(px, py, color=lm_color[name], s=150,
                       edgecolors='white', linewidth=1.2, zorder=5, marker='*')
            ax.text(px, py - 9, name, color='white', fontsize=8,
                    fontweight='bold', ha='center', va='bottom', zorder=6)
    else:
        for name, pa_flat, pb, score in all_matches[idx]:
            yb, xb = pb // w_p, pb % w_p
            px, py = (xb + 0.5) * PATCH_SIZE, (yb + 0.5) * PATCH_SIZE
            ax.scatter(px, py, color=lm_color[name], s=150,
                       edgecolors='white', linewidth=0.8, zorder=5, marker='*')
            # ax.text(px, py - 9, name, color='white', fontsize=8,
            #         fontweight='bold', ha='center', va='bottom', zorder=6)
        ax.set_title(f"{len(all_matches[idx])}/{len(lm_names)} lm", fontsize=7, pad=2)

plt.suptitle(
    f"Reference [{ROW_REF},{COL_REF}] — Facial Landmark Correspondences\n"
    f"(sim ≥ {SIM_THRESHOLD})  ·  Same color/label = same landmark across all portraits",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

which landmarks survive larger pose changes and which drift or drop
below the confidence threshold. Landmarks with more locally distinctive texture (eyes, nouse)
tend to be more robust than relatively featureless regions (a plain forehead or cheek), since the
matching has less to distinguish them from nearby patches.

One of DINOv2's more surprising properties: this kind of matching works **across different object
instances, and even different classes**, not just across pose of the same object — because the
features capture higher-level semantic structure, not just pixel-level appearance. Let's push that
further:

**Task 6:** rather than a handful of hand-picked landmark points, find correspondences between
**every** foreground patch across two *different* images — using **mutual nearest neighbors** (patch
A matches patch B only if each is the other's best match), restricted to foreground regions to avoid
spurious background matches.


In [ ]:
def mutual_nearest_neighbors(feats_a, feats_b, similarity_threshold=0.5):
    '''Find mutually-best-matching pairs between two sets of feature vectors.
    Returns (indices_into_a, indices_into_b, similarity_scores), sorted by score descending.'''
    # TODO: compute the cosine similarity matrix between feats_a and feats_b
    sim_matrix = ...              # (N_a, N_b)
    best_b_for_a = ...                     # best match in B, for each patch in A
    best_a_for_b = ...                     # best match in A, for each patch in B

    # TODO: find the mutually-best-matching pairs (i.e., A->B and B->A agree)        
    mutual_a = ... # indices of feats_a that have a mutual best match in feats_b
    mutual_b = ... # indices of feats_b that are the mutual best match for feats_a[mutual_a]
    scores = ... # similarity scores for the mutual matches
    
    keep = scores > similarity_threshold
    order = torch.argsort(scores[keep], descending=True)
    return mutual_a[keep][order], mutual_b[keep][order], scores[keep][order]


def foreground_patch_indices(coco_dataset, anns, patch_grid_shape):
    h, w, _ = patch_grid_shape
    full_mask = np.zeros((0, 0), dtype=bool)
    for a in anns:
        m = coco_dataset.coco.annToMask(a) > 0
        full_mask = m if full_mask.size == 0 else np.logical_or(full_mask, m)
    mask_t = torch.from_numpy(full_mask).float().unsqueeze(0).unsqueeze(0).to(device)
    patch_mask = F.interpolate(mask_t, size=(h, w), mode="nearest").squeeze() > 0.5
    return torch.nonzero(patch_mask.reshape(-1), as_tuple=False).squeeze(1), full_mask

example = 0
img_a_idx =  [474, 478, 4946, 2519][example] # COCO dataset index for Image A
img_b_idx =  [199, 489, 2532, 2489][example]  # COCO dataset index for Image B
top_k =  40  
img_a, anns_a = coco_dataset[img_a_idx]
img_b, anns_b = coco_dataset[img_b_idx]
patch_grid_a, pixel_a = get_patch_features(img_a)
patch_grid_b, pixel_b = get_patch_features(img_b)
h_a, w_a, dim = patch_grid_a.shape
h_b, w_b, _ = patch_grid_b.shape

fg_idx_a, mask_a = foreground_patch_indices(coco_dataset, anns_a, patch_grid_a.shape)
fg_idx_b, mask_b = foreground_patch_indices(coco_dataset, anns_b, patch_grid_b.shape)
feats_a_fg = patch_grid_a.reshape(-1, dim)[fg_idx_a]
feats_b_fg = patch_grid_b.reshape(-1, dim)[fg_idx_b]

mutual_a, mutual_b, scores = mutual_nearest_neighbors(feats_a_fg, feats_b_fg)
n_show = min(top_k, len(mutual_a))
flat_a = fg_idx_a[mutual_a[:n_show]].cpu().numpy()
flat_b = fg_idx_b[mutual_b[:n_show]].cpu().numpy()

points_a = [((idx % w_a + 0.5) * PATCH_SIZE, (idx // w_a + 0.5) * PATCH_SIZE) for idx in flat_a]
points_b = [((idx % w_b + 0.5) * PATCH_SIZE, (idx // w_b + 0.5) * PATCH_SIZE) for idx in flat_b]

disp_a, disp_b = to_rgb(pixel_a), to_rgb(pixel_b)
plot_correspondences(disp_a, disp_b, points_a, points_b,
                      title=f"Top-{n_show} mutual foreground correspondences")


These local features aren't only useful for finding correspondences — they can also serve as a
powerful backbone for dense prediction tasks like semantic or instance segmentation. You can go back
to the Week 6 tutorial and try improving those results using DINOv2 as the backbone instead.


---
# DINOv2 Tutorial Recap

We explored how DINOv2 represents visual information at two levels:

**1. Image-level features**
- DINOv2 produces image representations that are relatively invariant to nuisance variation
  (crops, color, blur, rotation) while staying sensitive to genuine semantic differences.
- These representations support image retrieval directly, and image classification with a linear
  probe trained in seconds on frozen features.

**2. Patch-level features**
- DINOv2 also gives us one feature vector per image patch — a dense, spatially structured
  representation.
- These features capture meaningful visual and semantic structure, visible through clustering.
- We used them for zero-shot object localization, landmark correspondence across pose, and dense
  correspondence between images. Correspondences can emerge even between different object instances
  and different classes, suggesting the features encode higher-level semantic structure, not just
  low-level appearance.

**The key takeaway:** DINOv2 is more than a feature extractor. Its pretrained representations
provide rich global *and* local visual features that can be reused across a wide range of
computer-vision tasks — often with little to no task-specific training.
